## Importing Packages

In [100]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset

import helper_utils


In [101]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [102]:
BATCH_SIZE = 32
SEED = 42
NUM_EPOCHS = 20

## Datasets and DataLoadoaders

### Dataset Path

In [103]:
dataset_path = "data"

class_names = ['Agricultural', 'Baseball Diamond', 'Buildings', 'Dense Residential',
               'Harbor', 'Medium Residential', 'Mobile Home Park', 'Parking Lot',
               'Runway', 'Sparse Residential', 'Storage Tanks', 'Tennis Court', 
               'Airplane', 'Beach', 'Chaparral', 'Forest', 'Freeway', 'Golf Course',
               'Intersection', 'Overpass', 'River'
              ]

### Dataset Transformations

In [104]:
mean = [0.485, 0.490, 0.451]
std = [0.214, 0.197, 0.191]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,std=std),
])

### Dataset Wrapper

In [105]:
class DenseDataset(Dataset):
    def __init__(self, data, transform = None):

        self.data = data
        self.transform = transform
        self.classes = data.dataset.classes

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):

        img, label = self.data[index]

        if self.transform:
            img = self.transform(img)

            return img, label

### Creating Datasets and DataLoaders

In [106]:
UC_Merced_dataset = ImageFolder(root = dataset_path, transform = None)

train_size = int(0.7 * (len(UC_Merced_dataset)))
test_size = int(0.15 * (len(UC_Merced_dataset)))
val_size = len(UC_Merced_dataset) - train_size - test_size

train_data, test_data, val_data = random_split(UC_Merced_dataset, [train_size, test_size, val_size], generator=torch.Generator().manual_seed(SEED))

train_dataset = DenseDataset(train_data, transform = train_transform)
test_dataset = DenseDataset(test_data, transform = val_transform)
val_dataset = DenseDataset(val_data, transform = val_transform)

num_classes = len(train_dataset.classes)

# Print dataset summary
print(f"Number of classes:      {num_classes}")
print(f"Training set size:      {len(train_dataset)}")
print(f"Validation set size:    {len(val_dataset)}")
print(f"Test set size:          {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle=False)  

Number of classes:      21
Training set size:      1470
Validation set size:    315
Test set size:          315


# Dummy Classes

In [107]:
class Bottleneck(nn.Sequential):
    pass

class FeatureExtractor(nn.Sequential):
    pass

class Transition(nn.Sequential):
    pass

class InitialBlock(nn.Sequential):
    pass

class DenseBlocks(nn.ModuleList):    
    pass

# Dense Layer Class

In [108]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate, bn_size):
        super(DenseLayer, self).__init__()

        self.in_channels = in_channels
        self.growth_rate = growth_rate

        # Bottleneck layer
        # 1 x 1 convolution to reduce the number of channels before the 3 x 3 convolution
        self.bottleneck = Bottleneck(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(in_channels = in_channels, out_channels = bn_size * growth_rate, kernel_size = 1, stride = 1, bias = False)
        )

        # Feature Extractor layer
        # 3 x 3 convolution to extract features
        self.feature_extractor = FeatureExtractor(
            nn.BatchNorm2d(bn_size * growth_rate),
            nn.ReLU(inplace = True),
            nn.Conv2d(in_channels = bn_size * growth_rate, out_channels = growth_rate, stride = 1, kernel_size = 3, padding = 1, bias = False)
        )


    def forward(self, x):

        new_features = self.bottleneck(x)
        new_features = self.feature_extractor(new_features)

        x = torch.cat([x, new_features], dim = 1)

        return x

## Dense Block Class

In [109]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate, bn_size):

        super(DenseBlock, self).__init__()

        self.layers = nn.ModuleList()

        for i in range(num_layers):

            layer = DenseLayer(in_channels = in_channels + (i * growth_rate), growth_rate = growth_rate, bn_size = bn_size)

            self.layers.append(layer)

    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)

        return x



## Transition Layer

In [110]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression_factor = 0.5):
        super(TransitionLayer, self).__init__()

        out_channels = int(in_channels * compression_factor)

        self.transition = Transition(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace = True),

            # 1 x 1 convolution to reduce the number of channels by the compression factor. 
            nn.Conv2d(in_channels = in_channels, out_channels = out_channels, kernel_size = 1, stride = 1, bias = False),

            # Average pooling to reduce the spatial dimensions by half.
            nn.AvgPool2d(kernel_size = 2, stride = 2)
        )

    
    def forward(self, x):
        x = self.transition(x)

        return x

## DenseNet Class

In [111]:
class DenseNet(nn.Module):
    def __init__(self, num_classes = 1000, growth_rate = 32, num_init_features = 64, block_config = (6, 12, 24, 16), bn_size = 4, compression_factor = 0.5):

        super(DenseNet, self).__init__()


        self.initial_features = self._get_initial_features(num_init_features)

        self.dense_blocks = self._get_blocks(
            num_init_features = num_init_features, 
            growth_rate = growth_rate, 
            block_config = block_config, 
            bn_size = bn_size, 
            compression_factor = compression_factor)
        
        self.classifier = nn.Linear(in_features = self.num_features, out_features = num_classes)


    def _get_initial_features(self, num_init_features):

        initial_block = InitialBlock(
            nn.Conv2d(
                in_channels = 3,
                out_channels = num_init_features,
                kernel_size = 7, 
                stride = 2,
                padding = 3,
                bias = False 
            ),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace = True),
            nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        )

        return initial_block
        
    def _get_blocks(self, num_init_features, growth_rate, block_config, bn_size, compression_factor):
        
        dense_blocks = DenseBlocks()

        num_features = num_init_features

        for i, num_layers in enumerate(block_config):
            
            block = DenseBlock(num_layers = num_layers, in_channels = num_features, growth_rate = growth_rate, bn_size = bn_size)
            dense_blocks.append(block)

            num_features = num_features + (num_layers * growth_rate)

            if i != len(block_config) - 1:
                transition = TransitionLayer(in_channels=num_features, compression_factor=compression_factor)
                dense_blocks.append(transition)
        
                num_features = int(num_features * compression_factor)


        # Add the final layers
        dense_blocks.append(nn.BatchNorm2d(num_features))
        dense_blocks.append(nn.ReLU(inplace=True))
        dense_blocks.append(nn.AdaptiveAvgPool2d((1, 1)))

        # Store the final number of features for the classification layer
        self.num_features = num_features

        return dense_blocks
        


    def forward(self, x):
        x = self.initial_features(x)

        for block in self.dense_blocks:
            x = block(x)

        # Flatten the output tensor for the fully connected layer.
        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

                    

In [112]:
torch.manual_seed(SEED)

densenet_model = DenseNet(num_classes=num_classes)
input_size = (BATCH_SIZE, 3, 224, 224)

In [113]:
# Define a configuration dictionary to store parameters for the model summary.
config = {
    "input_size": input_size, # (batch_size, *img_shape)
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": 2
}

# Generate the model summary object using torchinfo with the specified configuration.
summary = torchinfo.summary(
    model=densenet_model, 
    input_size=config["input_size"], 
    col_names=config["attr_names"], 
    depth=config["depth"]
)

# Display the summary as a styled HTML table.
print("--- Model Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Model Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
DenseNet (DenseNet),"[32, 3, 224, 224]","[32, 21]",--
InitialBlock (initial_features): 1-1,"[32, 3, 224, 224]","[32, 64, 56, 56]",--
Conv2d (0): 2-1,"[32, 3, 224, 224]","[32, 64, 112, 112]","9,408"
BatchNorm2d (1): 2-2,"[32, 64, 112, 112]","[32, 64, 112, 112]",128
ReLU (2): 2-3,"[32, 64, 112, 112]","[32, 64, 112, 112]",--
MaxPool2d (3): 2-4,"[32, 64, 112, 112]","[32, 64, 56, 56]",--
DenseBlocks (dense_blocks): 1-2,[],[],--
DenseBlock (0): 2-5,"[32, 64, 56, 56]","[32, 256, 56, 56]","335,040"
TransitionLayer (1): 2-6,"[32, 256, 56, 56]","[32, 128, 28, 28]","33,280"
DenseBlock (2): 2-7,"[32, 128, 28, 28]","[32, 512, 28, 28]","919,680"


In [114]:
loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(densenet_model.parameters(), lr=0.001)

In [ ]:
trained_densenet, history, confusion_matrix = helper_utils.training_loop_16_mixed(
    model=densenet_model,
    train_loader= train_loader,
    val_loader= val_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    num_epochs= NUM_EPOCHS,
    device=device,
    save_path='/tmp/saved_models/best_trained_densenet.pth',
)

Overall Progress:   0%|          | 0/1120 [00:00<?, ?it/s]